**Tokenize/Encode Reviews - Word2Vec**

First we install any dependencies we might need, then connect to Google Drive.

In [ ]:
pip install nltk gensim

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Now we import libraries, specify our Word2Vec model (specifically the Google News version with 300d embeddings), and define a function to iterate through a JSON and compute the word average vector for the text of each review. We round each value to 4 places to cut down substantially on data size  (over 50% smaller) without losing any real functional capacity for our application. It doesn't scale exactly because of other fields in the json files, such as ids.

In [ ]:
# Imports
import json
import numpy as np
import gensim.downloader as api
from nltk.tokenize import word_tokenize
import nltk
import time

# Download tokenizer
nltk.download('punkt')
nltk.download('punkt_tab')

# Load Word2Vec via Gensim downloader
w2v = api.load('word2vec-google-news-300')
D   = w2v.vector_size                        # 300

# Define processing function
def process_review_word_average(input_path, output_path, categories=False):
    with open(input_path,  'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:

        for line in fin:
            review = json.loads(line)
            if categories:
              text = review.get('categories', '')
            else:
              text = review.get('text', '')

            # tokenize & lowercase
            tokens = [t.lower() for t in word_tokenize(text)]

            # keep only tokens in vocab
            tokens = [t for t in tokens if t in w2v.key_to_index]

            # compute average (zeros if none)
            if tokens:
                mat     = np.stack([w2v[t] for t in tokens])
                avg_vec = mat.mean(axis=0)
            else:
                avg_vec = np.zeros(D, dtype=float)

            # replace string with new embedding in output (same field name for reference)
            # round to 5 places to minimize storage needs (default seems to be ~18 places)
            review['text'] = [round(float(x), 4) for x in avg_vec]
            fout.write(json.dumps(review) + '\n')

    print("Embeddings saved to:", output_path)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[==================================================] 100.0% 1662.8/1662.8MB downloaded


Finally, we execute this script on the review files as well as the category tags

In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/user_tower_reviews.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Word2Vec/user_tower_reviews.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_word_average(input_path = INPUT_PATH, output_path = OUTPUT_PATH)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Embeddings saved to: /content/drive/MyDrive/Colab_Folder/266_Project/Data/Word2Vec/user_tower_reviews.json
Time taken: 134.17177033424377 seconds


In [ ]:
# File paths (adjust as needed)
INPUT_PATH  = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Raw/business_category_tags.json'
OUTPUT_PATH = '/content/drive/MyDrive/Colab_Folder/266_Project/Data/Word2Vec/business_category_tags.json'

# Run the processing on each review file and category tags
start_time = time.time()
process_review_word_average(input_path = INPUT_PATH, output_path = OUTPUT_PATH, categories = True)
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Embeddings saved to: /content/drive/MyDrive/Colab_Folder/266_Project/Data/Word2Vec/business_category_tags.json
Time taken: 28.56012988090515 seconds


Execution times were on high RAM CPU only instance.